# 04. 최종 모델

**최종 결과**: Public F1 0.44897 (43위) → Private F1 0.41071 (26위 / 733팀, 상위 3.5%)

## 구조 요약

```
1. Rule-based Scoring (get_refined_score)
   └─ 재등록, 페르소나, 키워드, Hard Kill

2. 5-Seed × 5-Fold Stacking Ensemble
   ├─ CatBoost (범주형)
   ├─ LightGBM (수치형)
   └─ Meta LR (OOF 스태킹)

3. Hard Kill Rules
   └─ 법학 / 경영학×직장인·취준생 → prob=0

4. Top-375 Selection (Flip-rate 46.1%)
   └─ Threshold 방식 대신 순위 기반 고정 선발
```

## 핵심 설계 원칙
- **L1 < 0.2 변수만** 사용 (Train/Test 이질성 대응)
- **재등록**이 유일하게 4조건 통과한 안정 신호 (Lift=1.38, Support=146, L1=0.010)
- **Flip-rate**: 확률 분포 밀집 구간(0.35~0.40)에서 Threshold 불안정 → 순위 고정 선발로 해결

## 1. 초기화 및 데이터 로드

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')
TARGET = 'completed'
y = train[TARGET].astype(int).values

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Baseline 수료율: {y.mean():.3f} ({y.mean()*100:.1f}%)")

## 2. Rule-based Scoring

03_lift_analysis에서 발견한 패턴을 점수로 변환.

### 사용 변수 선택 근거
| 변수 | L1 | 비고 |
|------|-----|------|
| re_registration | 0.010 | 유일하게 4조건 통과 (Lift=1.38, Support=146) |
| job | 0.075 | 페르소나 조합용 |
| major_field | 0.421 | 주의 — 특정 카테고리(경영학·법학 Kill)만 사용 |
| time_input | 연속형 | 분포 유사 |
| whyBDA + incumbents_lecture_scale_reason | 합산 | incumbents L1=1.788이나 공통 패턴 존재 |

### 키워드 분석 결과 (03_lift_analysis 기준)
- '기회': Lift=1.28, Support=21 — 합산 텍스트 기준 양의 신호
- '함께': Lift=1.01, Support=10 — 미약한 신호
- '혼자', '어려워': Lift<1 — 부정 신호 (맥락 고려 필요)

In [ ]:
def get_refined_score(row):
    """
    도메인 지식 기반 점수 시스템.

    설계 원칙:
    - 안정 변수(L1<0.2) 우선 사용
    - 부정 키워드는 맥락 고려 ("혼자 공부하기 어려워서 함께 배우고 싶어요" → 감점 없음)
    - 법학/경영학×직장인·취준생은 0% 수료율 패턴 → Hard Kill (Step 4에서 처리)
    """
    t1 = str(row['whyBDA']) if not pd.isna(row['whyBDA']) else ""
    t2 = str(row['incumbents_lecture_scale_reason']) if not pd.isna(row['incumbents_lecture_scale_reason']) else ""
    full_text = t1 + " " + t2
    score = 0

    major = str(row['major_field'])
    job   = str(row['job'])
    desired_job = str(row['desired_job'])

    # ── 페르소나 (Lift > 1.3이나 Support 부족 → 도메인 판단으로 적용) ──────────
    # 자연과학×취준생: Lift=1.86, Support=9 (표본 부족)
    if "자연과학" in major and "취준생" in job:
        score += 3
    # 인문학×대학생: Lift=1.54, Support=37 (표본 부족)
    if "인문학" in major and "대학생" in job:
        score += 2
    # 데이터분석가 희망×온라인 선호
    if "데이터 분석가" in desired_job and "온라인" in str(row['incumbents_lecture_type']):
        score += 2

    # ── 긍정 키워드 (합산 텍스트 기준) ─────────────────────────────────────────
    if "함께" in full_text:
        score += 1
    if "기회" in full_text:
        score += 3

    # ── 부정 키워드 — 맥락 고려하여 중복 감점 방지 ─────────────────────────────
    # 예: "혼자 공부하기 어려워서 함께 배우고 싶어요" → 긍정맥락 있으므로 감점 없음
    # 예: "혼자 공부하기 어려워서" (단순 응답) → -2점
    if "경험" in full_text:
        score -= 2
    positive_context = ["기회", "함께", "도움", "필요", "배우", "성장", "발전"]
    has_positive = any(word in full_text for word in positive_context)
    if ("어려워" in full_text or "혼자" in full_text) and not has_positive:
        score -= 2  # 통합 -2점 (중복 감점 방지)

    # ── 방어 규칙 ────────────────────────────────────────────────────────────────
    inflow = str(row['inflow_route'])
    if "인스타그램" in inflow and (len(t1) + len(t2)) < 30:
        score -= 3  # 짧은 응답 + 인스타 유입 → 성의 없음

    # ── 안정 신호 (L1<0.2, 4조건 통과) ─────────────────────────────────────────
    if row['time_input'] == 4.0:
        score += 2
    if row['re_registration'] == '예':   # Lift=1.38, Support=146, L1=0.010
        score += 2

    # ── school1 보정 (L1=0.718 불안정 — Train EDA 기반 제한적 적용) ─────────────
    # Train에서 수료율 60%+ 학교 (n≥15): 45, 38, 52
    # Train에서 수료율 0~10% 학교 (n≥10): 81, 54, 19
    # school1은 L1 높아 신뢰도 낮음 — 실험적으로 소폭 효과 확인
    school = str(row['school1'])
    if school in ['45', '38', '52']:
        score += 2
    if school in ['81', '54', '19']:
        score -= 5

    return score

# 점수 분포 확인
train['refined_score'] = train.apply(get_refined_score, axis=1)
test['refined_score']  = test.apply(get_refined_score, axis=1)

print("Train refined_score 분포:")
print(train['refined_score'].describe())
print("\nTest refined_score 분포:")
print(test['refined_score'].describe())

## 3. Feature Engineering

In [ ]:
for df in [train, test]:
    df['is_non_major']   = (~df['major_data']).astype(int)
    df['is_double_major'] = df['major type'].str.contains('복수|다중|이중', na=False).astype(int)
    # 텍스트 응답 길이 (whyBDA + incumbents_lecture_scale_reason 합산)
    df['len_total'] = (
        df['whyBDA'].fillna('').apply(len)
        + df['incumbents_lecture_scale_reason'].fillna('').apply(len)
    )

TARGET = 'completed'
X_tr = train.drop(columns=[TARGET, 'ID']).copy()
X_te = test.drop(columns=['ID']).copy()

cat_cols = [c for c in X_tr.columns if X_tr[c].dtype == 'object']
for c in cat_cols:
    X_tr[c] = X_tr[c].fillna('NAN').astype(str)
    X_te[c] = X_te[c].fillna('NAN').astype(str)

X_tr_num = X_tr.select_dtypes(exclude=['object'])
X_te_num = X_te.select_dtypes(exclude=['object'])
cat_features_idx = [i for i, c in enumerate(X_tr.columns) if c in cat_cols]

print(f"Feature 수: {X_tr.shape[1]} (cat: {len(cat_cols)}, num: {X_tr_num.shape[1]})")

## 4. 5-Seed × 5-Fold Stacking Ensemble

```
For each seed in [41, 42, 43, 44, 45]:
  ├─ CatBoost (범주형 포함) → OOF prob
  ├─ LightGBM (수치형만)   → OOF prob
  └─ Meta LR (OOF 스태킹) → Test prob

최종 = 5개 seed 평균
```

**설계 이유**: 단일 모델은 확률 분포가 0.35~0.40 구간에 밀집 → Threshold 불안정  
→ 앙상블로 확률 분산 확대, Flip-rate 선발 안정화

In [ ]:
seeds = [41, 42, 43, 44, 45]
final_ensemble_prob = np.zeros(len(X_te))

for seed in seeds:
    set_seed(seed)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    oof_lgbm, oof_cat = np.zeros(len(X_tr)), np.zeros(len(X_tr))
    test_lgbm, test_cat = np.zeros(len(X_te)), np.zeros(len(X_te))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr, y)):
        # ── CatBoost ────────────────────────────────────────────────────────────
        cat = CatBoostClassifier(
            iterations=2000, learning_rate=0.03, depth=6,
            random_seed=seed, verbose=0, early_stopping_rounds=100
        )
        cat.fit(
            Pool(X_tr.iloc[tr_idx], y[tr_idx], cat_features=cat_features_idx),
            eval_set=Pool(X_tr.iloc[va_idx], y[va_idx], cat_features=cat_features_idx)
        )
        oof_cat[va_idx]  = cat.predict_proba(X_tr.iloc[va_idx])[:, 1]
        test_cat        += cat.predict_proba(X_te)[:, 1] / 5

        # ── LightGBM ─────────────────────────────────────────────────────────────
        dtrain = lgb.Dataset(X_tr_num.iloc[tr_idx], y[tr_idx])
        dval   = lgb.Dataset(X_tr_num.iloc[va_idx], y[va_idx], reference=dtrain)
        lgbm_model = lgb.train(
            {'objective': 'binary', 'metric': 'binary_logloss',
             'learning_rate': 0.03, 'seed': seed, 'verbose': -1},
            dtrain, 2000, valid_sets=[dval],
            callbacks=[lgb.early_stopping(100, verbose=False)]
        )
        oof_lgbm[va_idx] = lgbm_model.predict(X_tr_num.iloc[va_idx])
        test_lgbm       += lgbm_model.predict(X_te_num) / 5

    # ── Meta Model (OOF 스태킹) ─────────────────────────────────────────────────
    meta = LogisticRegression(random_state=seed)
    meta.fit(pd.DataFrame({'LGBM': oof_lgbm, 'CAT': oof_cat}), y)
    final_ensemble_prob += meta.predict_proba(
        pd.DataFrame({'LGBM': test_lgbm, 'CAT': test_cat})
    )[:, 1] / len(seeds)

    print(f"  Seed {seed} 완료")

print(f"\nEnsemble prob 통계: mean={final_ensemble_prob.mean():.3f}, std={final_ensemble_prob.std():.3f}")

## 5. Hard Kill Rules

Train 데이터에서 수료율이 0%에 가까운 집단을 확률 0으로 강제 처리.

| Rule | Train 수료율 | Support | L1 |
|------|------------|---------|-----|
| 법학 관련 전공 | ~0% | 4명 | 0.421 |
| 경영학 × 직장인 | 0% | 5명 | 0.421 |

> **주의**: major_field L1=0.421 (보통 수준), Support 부족 — 강한 패턴이나 일반화 한계 존재

In [ ]:
test_origin = pd.read_csv('../data/test.csv')
fail_indices = []

# 법학 Kill
fail_indices.extend(test_origin[
    test_origin['major1_1'].str.contains('법', na=False) |
    (test_origin['major_field'] == '법학')
].index)

# 경영학 × 직장인·취준생 Kill
fail_indices.extend(test_origin[
    (test_origin['major_field'] == '경영학') &
    (test_origin['job'].isin(['직장인', '취준생']))
].index)

kill_set = set(fail_indices)
final_ensemble_prob[list(kill_set)] = 0.0
print(f"Hard Kill 적용: {len(kill_set)}명 제거")

## 6. Top-375 Flip-rate Selection

**Threshold 방식의 문제**: 확률이 0.35~0.40 구간에 밀집 → Threshold 0.005 차이로 15~25명 변동  
**해결책**: "몇 명 선발?" 기준으로 전환 → 순위 기반 고정 선발

- 선발 인원: **375명** (Flip-rate = 375/814 = 46.1%)
- tie-breaking: 텍스트 응답 길이(len_total)로 동점자 처리

In [ ]:
# Tie-breaking: 텍스트 응답이 길수록 성의 있음 → 미세 보정
final_ensemble_prob += test['len_total'] * 0.0000001

TARGET_CNT = 375
threshold = np.sort(final_ensemble_prob)[::-1][TARGET_CNT - 1]
final_pred = (final_ensemble_prob >= threshold).astype(int)

assert final_pred.sum() == TARGET_CNT, f"선발 인원 불일치: {final_pred.sum()}"
print(f"선발 인원: {final_pred.sum()}명 / {len(final_pred)}명 ({final_pred.sum()/len(final_pred)*100:.1f}%)")

## 7. 제출 파일 저장

In [ ]:
save_path = '../outputs/FINAL_NLP_V2_375.csv'
pd.DataFrame({'ID': test['ID'], 'completed': final_pred}).to_csv(save_path, index=False)
print(f"저장 완료: {save_path}")

## 8. 최종 결과 요약

| 지표 | 값 |
|------|----|
| Public F1 | **0.44897** (43위 / 733팀) |
| Private F1 | **0.41071** (26위 / 733팀, 상위 3.5%) |
| Public → Private 순위 변화 | 43위 → 26위 (+17등) |

### Public → Private +17등 상승의 의미
- Public LB 과적합을 하지 않은 접근이 Private에서 더 일반화됨
- 0.4490 달성 후 실험 중단 → 안정적 신호만 유지한 보수적 전략
- L1 기반 변수 선별이 9기→10기 분포 이동에 강건하게 작동

### 1위 팀과의 공통점/차이점
| | 1위 팀 | 본 모델 |
|--|--------|--------|
| Flip-rate | 46.9% | 46.1% (375/814) |
| 보정 방식 | Rescue-Demote (LR→CB cascade) | Rule Score + Hard Kill |
| 피처 수 | 16개 (최소화) | Rule Score + FE |

→ **Flip-rate 전략 독립적으로 동일하게 발견**